# Fase C2 · Prueba de `alpha_max` en CEC2013 — Mac mini

La sensibilidad B4 (50k) mostró que `alpha_max = 0.9` mejora mucho al valor por defecto 0.5. Pero el
0.5 se eligió porque 0.99 colapsaba la colonia, y eso debería notarse a presupuestos largos. Esta prueba
mira si la mejora se sostiene a **500k** antes de decidir si se recalibra (y se repite el protocolo).

Mismo camino que B4: A2_hgt, CEC2013 (28 funciones, suite que no se usa para evaluar), D10 y D50,
15 corridas.
- **50k:** default (tiene que reproducir B4 bit a bit), `alpha_max` 0.95 y 0.99, combo
- **500k:** default, `alpha_max` 0.7, 0.9 y 0.99, combo
- **combo** = `alpha_max` 0.9 + `qs_gain` 4 + `N0` 40 (los tres que mejoraron a D50 en B4)

Tiempo aprox.: **~7 h** (50k ~30 min; 500k ~1,5 h en D10 y ~5 h en D50). Se puede dejar de noche.
Copiar al mini: **`prueba_alpha.py`** (nuevo, commit `bb9765b`). Necesita los `SENS_default_*.json` de B4.
No correrlo al mismo tiempo que `faseC_clonalg` (los dos usan 20 workers).
**Solo en el Mac mini.**

In [ ]:
import sys, os, hashlib, subprocess, opfunu

WORKERS = 20
ESPERADO = {
    "prueba_alpha.py":        "2e8316802bb9432409f13060f32be666527252caab238004bcff4158ddaeeea2",
    "complementarias.py":     "2889b74aae020d143a4980e19a70ccc4c6e959dceb379f91e178c681cee8af1a",
    "marea2.py":              "10d7448e59082adf4f464068646d74d25d8b8f54f5317f9eb4dfed32dacfe64b",
    "experimento_config.py":  "ef9c5af25d46afa681276efeb5d0e25c68290404dc6093e8d8783b3554a914d1",
}

def correr(*args):
    """Ejecuta un script con el mismo Python del kernel y muestra la salida en vivo."""
    p = subprocess.Popen([sys.executable, *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        if "pkg_resources" not in linea:
            print(linea, end="")
    return p.wait()

print("python :", sys.executable)
print("carpeta:", os.getcwd())
print("opfunu :", opfunu.__version__)
assert opfunu.__version__ == "1.0.4", "opfunu tiene que ser 1.0.4 (1.0.1 esta roto)"
for f, h in ESPERADO.items():
    real = hashlib.sha256(open(f, "rb").read()).hexdigest()
    print(f"{f:24s} {real[:16]}  {'OK' if real == h else 'DISTINTO'}")
    assert real == h, f"{f} no es la version del commit bb9765b: copiala de nuevo"
print("Entorno y archivos OK")

## 1 · Chequeo (segundos)
Reproduce una corrida de B4 y prueba la combinación.

In [ ]:
rc = correr("prueba_alpha.py", "check")
assert rc == 0, "el chequeo fallo: no seguir, mandar la salida a Claude"
CHEQUEADO = True

## 2 · Correr (~7 h; primero 50k, después 500k)

In [ ]:
assert globals().get("CHEQUEADO"), "primero la celda de chequeo"
rc = correr("prueba_alpha.py", "correr", "--workers", str(WORKERS))
assert rc == 0

## 3 · Verificar y traer al MacBook
Tiene que terminar en **TODO OK** (incluye que el default a 50k reproduzca B4).
Traer los 18 `resultados_experimento/complementarias/ALPHA_*.json` a esa misma carpeta del MacBook
(**no a la raíz**).

In [ ]:
rc = correr("prueba_alpha.py", "verificar")
print("\nverificacion:", "OK" if rc == 0 else "REVISAR")